# 01 Vorbereitung — Kalibrierung, Plate-Solve, Vergleichssterne

**Wann:** nach der Aufnahme; die Bilder liegen im Sitzungsordner (`lights/`, `darks/`, `flats/`, `biases/`). **Voraussetzung:** NB00 ist gelaufen (die **Sitzung** ist angelegt).
**Ergebnis:** kalibrierte Bilder, ein gelöstes **Referenzbild** (WCS) und die **Comp-/Check-Sterne** (`field_config.json`) — alles in der Sitzung gespeichert.
**Warum:** Ohne Kalibrierung (Dunkelstrom, Flat) und ohne Koordinaten je Bild lässt sich weder messen noch die Lichtkurve einem Katalog zuordnen.

### So bedienen Sie dieses Notebook

| | |
|---|---|
| **1. Formular anzeigen** | Die Zelle unter „Formular“ ausführen (▶ links neben der Zelle oder `Umschalt+Eingabe`) — die Eingabefelder erscheinen *darunter*. |
| **2. Prüfen / ausfüllen** | Meist ist nichts zu ändern; Felder siehe Tabelle. **Oben im Formular erscheint nach jeder Auswahl sofort ein Hinweis** (grün = passt, blau = zur Orientierung, orange = prüfen, rot = so geht es nicht). Erst danach weiter. |
| **3. Automatisch** | Ab der Zelle **„Ab hier automatisch“**: Pfeil neben der Zelle → **„Diese Zelle und alle folgenden“** ausführen. Es ist nichts mehr einzugeben. |
| **Code zuklappen** | Sie müssen den Code nicht lesen. VS Code: Befehlspalette (`Umschalt+⌘+P`) → **„Notebook: Collapse All Cell Inputs“**. |

### Was in Ihrem Fall passiert (Fall aus NB00 Schritt 1)

| Fall | Kalibrierung (Siril) | Danach |
|---|---|---|
| **A  Monochrom + Standardfilter** | normal: Bias, Dunkel, Flat → `process/pp_light_*.fit` | weiter mit NB02 |
| **B  Monochrom + RGB-Filterrad** | wie A; **Flats passend zum jeweiligen Filter** | NB02 je Filter, dann NB08 |
| **C  Farbkamera (OSC)** · **D  DSLR (RAW)** | **im CFA-Bereich, ohne Debayern** (`-cfa -equalize_cfa`); RAW wird zuerst ohne Debayern in FITS gewandelt | CFA in R/G/B trennen (`python tools/split_cfa.py <Ordner> <Ausgabe>`), NB02 je Kanal, dann NB08 |

Haben Sie schon kalibrierte Bilder (z. B. LCO), im Formular **„Bilder sind schon kalibriert“** wählen.

### Formular — was eintragen

**1. Zelle unten ausführen (▶) — das Formular erscheint darunter. 2. Prüfen wie in der Tabelle (meist ist nichts zu ändern). 3. Weiter mit „Ab hier automatisch“.**

| Feld | **Testlauf** (LCO-Testdaten) | **Eigene Aufnahmen** |
|---|---|---|
| **Kalibrierung** | schon vorgewählt: *Bilder sind schon kalibriert* — so lassen | *Siril-Skript erzeugen und ausführen* (Standard); *nur erzeugen*, wenn Sie es in Siril selbst starten; *schon kalibriert* nur, wenn Sie das selbst getan haben |
| **Ordner der Bilder** | leer lassen | leer lassen (= `process/` der Sitzung); nur ändern, wenn die Bilder woanders liegen |
| **Referenzbild** | leer lassen | leer = scharfes Bild aus NB00; sonst ein eigenes angeben |
| **ASTAP-Radius [°]** | 10 lassen | 10 lassen; bei Weitfeld (> 10° Feld) größer |
| **Sequenz bis V =** | 11 lassen | Helligkeitsgrenze der Vergleichssterne; bei schwachen Zielen höher |
| **Comps mit Gaia auffüllen** | Häkchen lassen | Häkchen lassen; Gaia-Werte dienen nur als Hilfe (keine Transformation) |
| **Gaia-Gegenprüfung** | Häkchen lassen (braucht Netz) | Häkchen lassen (braucht Netz) |
| **Regeln (JSON)** | leer lassen | nur für Fortgeschrittene, z. B. `{"comp_bv_tol": 0.7}` |

In [ ]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

from varphot import Session, load_plan
from varphot import workflow as W
from varphot.hints import bind, nb01_hints
from varphot.siril import find_siril

SESS = Session.current()
PLAN = load_plan(SESS.plan) if SESS.plan and Path(SESS.plan).exists() else None
print(f"Sitzung {SESS.workspace}\n  Ziel {SESS.target} · Methode {SESS.method} · Daten: {SESS.source}" + ("" if PLAN else "\n  ⚠ kein Plan — NB00 ausführen (Aperturen, Filter, Comp-Regeln fehlen sonst)"))

_st, _lay = {"description_width": "150px"}, widgets.Layout(width="620px")
w_cal = widgets.Dropdown(description="Kalibrierung:", options=[("Siril-Skript erzeugen und ausführen", "run"), ("Siril-Skript nur erzeugen (selbst ausführen)", "script"), ("Bilder sind schon kalibriert", "done")],
                         value=os.environ.get("VARPHOT_CAL", "done" if SESS.source == "lco-demo" else "run"), style=_st, layout=_lay)
w_frames = widgets.Text(description="Ordner der Bilder:", value=os.environ.get("VARPHOT_FRAMES_DIR") or SESS.frames_dir, placeholder="leer = process/ der Sitzung", style=_st, layout=_lay)
w_ref = widgets.Text(description="Referenzbild:", value=os.environ.get("VARPHOT_REF", ""), placeholder="leer = scharfes Bild aus NB00, sonst das beste der Reihe", style=_st, layout=_lay)
w_radius = widgets.BoundedFloatText(description="ASTAP-Radius [°]:", value=10.0, min=1.0, max=30.0, style=_st, layout=_lay)
w_maglimit = widgets.FloatSlider(description="Sequenz bis V =", value=float(os.environ.get("VARPHOT_MAGLIMIT", 11.0)), min=8.0, max=14.0, step=0.5, style=_st, layout=_lay)
w_gaia_fill = widgets.Checkbox(description="Comps mit Gaia-Feldsternen auffüllen, wenn die AAVSO-Sequenz zu klein ist", value=True, indent=False, layout=_lay)
w_gaia_check = widgets.Checkbox(description="Gaia-Gegenprüfung (WCS und Sequenzsterne; braucht Netz)", value=os.environ.get("VARPHOT_GAIA_CHECK", "1") != "0", indent=False, layout=_lay)
w_rules = widgets.Text(description="Regeln (JSON):", value=os.environ.get("VARPHOT_RULES", "{}"), placeholder='z. B. {"comp_bv_tol": 0.7}', style=_st, layout=_lay)
w_hint = widgets.HTML()
def _ref_ok():
    r = w_ref.value.strip()
    return bool(r) and (Path(r).exists() or (Path(SESS.workspace) / r).exists())
bind(w_hint, [w_cal, w_ref, w_radius, w_gaia_fill, w_gaia_check],
     lambda: nb01_hints(w_cal.value, find_siril() is not None, SESS.method, w_ref.value, _ref_ok(), w_radius.value, w_gaia_fill.value, w_gaia_check.value, lco=SESS.source == "lco-demo"))
display(widgets.VBox([w_hint, w_cal, w_frames, w_ref, w_radius, w_maglimit, w_gaia_fill, w_gaia_check, w_rules]))

## Ab hier automatisch

*Ausführen → Diese Zelle und alle folgenden.* Stoppt mit einer klaren Meldung, wenn etwas fehlt (z. B. Siril-Skript noch nicht gelaufen).

In [ ]:
class Stop(Exception):
    """Sachlicher Abbruch mit Anleitung (kein Programmfehler)."""

if w_cal.value in ("run", "script"):
    CAL = W.calibration_step(SESS, run=(w_cal.value == "run"))
    if CAL["status"] == "keine_lights":
        raise Stop("lights/ der Sitzung ist leer: Light-Bilder (und Darks/Flats/Biases) in die Unterordner legen und erneut starten — oder oben „Bilder sind schon kalibriert“ wählen.")
    print(f"Aufnahmearten: {CAL['kinds']}")
    if CAL["status"] == "skript":
        raise Stop(f"Siril-Skript liegt bereit: {CAL['script']}\n  Ausführen: siril-cli -d \"{SESS.workspace}\" -s \"{CAL['script']}\"  (oder in der Siril-Oberfläche: Skripte → Skript ausführen, Arbeitsordner = Sitzungsordner).\n"
                   "  Danach diese Zelle und die folgenden erneut ausführen." + (f"\n  ({CAL['log']})" if CAL["log"] else ""))
    if CAL["status"] == "fehler":
        raise Stop("Siril lief, aber es entstanden keine kalibrierten Bilder (process/pp_light_*). Protokoll:\n" + CAL["log"][-1500:])
    SESS.update(frames_dir=str(SESS.path("process")))
    print(f"Kalibriert: {len(CAL['frames'])} Bilder in process/")
else:
    SESS.update(frames_dir=w_frames.value.strip() or SESS.frames_dir or str(SESS.path("process")))
FRAMES = W.collect_frames(SESS)
if not FRAMES:
    raise Stop(f"Keine Bilder in {SESS.frames_dir}. Ordner prüfen (FITS, bei Siril: process/pp_light_*).")
print(f"{len(FRAMES)} Bilder der Reihe in {SESS.frames_dir}")

In [ ]:
if w_ref.value.strip():
    REF = Path(w_ref.value.strip()).expanduser()
elif SESS.ref_frame and Path(SESS.ref_frame).exists():
    REF = Path(SESS.ref_frame)
    print("Referenzbild: das scharfe Bild aus NB00")
else:
    REF, REFTAB = W.pick_reference(FRAMES)
    print("Referenzbild automatisch gewählt (meiste scharfe Sterne):"); display(REFTAB.round(2))
RES, SUMM = W.solve_reference(SESS, REF, radius_deg=w_radius.value, fov_deg=(min(PLAN.field_deg) if PLAN and PLAN.field_deg else None))
IMG_R, HDR_R, WCS_R = W.load_reference(SESS)
print(f"Referenz {REF.name}: WCS aus {RES.source}, Skala {SUMM.get('scale_arcsec', '?')} ″/px" + ("" if RES.source == "ASTAP" else "  ⚠ ASTAP hat nicht gelöst — WCS aus dem FITS-Kopf (ungeprüft)"))
for n in RES.notes:
    print("  ⚠", n)
QUAL = W.wcs_quality(WCS_R, IMG_R, HDR_R) if (w_gaia_check.value and RES.source == "ASTAP") else dict(checked=False, note="übersprungen")
if QUAL["checked"]:
    print(f"Gaia-Gegenprobe: Versatz {QUAL['offset_arcsec']:.2f}″, Streuung {QUAL['rms_arcsec']:.2f}″ ({QUAL['n_match']} Treffer) → {QUAL['verdict']}")
else:
    print("Gaia-Gegenprobe:", QUAL["note"])

In [ ]:
FIELD = W.select_field(SESS, PLAN, maglimit=w_maglimit.value, use_gaia_fill=w_gaia_fill.value, rules=json.loads(w_rules.value or "{}"), gaia_check=w_gaia_check.value)
for w in FIELD["warnings"]:
    print("⚠", w)
res = FIELD["res"]
cols = [c for c in ("role", "label", "V", "V_err", "BV", "sep_deg", "reasons") if c in res.colnames]
tab = res[cols].to_pandas()
tab["role"] = pd.Categorical(tab["role"], ["comp", "check", "reserve", "verworfen"], ordered=True)
display(tab[tab["role"] != "verworfen"].sort_values(["role", "sep_deg"]).round(3).reset_index(drop=True))
print("Rollen:", FIELD["counts"], f"| Sequenz-ID (CHART): {FIELD['record']['vsp'].get('chartid')}")
from varphot import plot_comp_field
plot_comp_field(res, FIELD["target"], field_deg=FIELD["field_deg"], pa_deg=FIELD["pa_deg"]); plt.show()

In [ ]:
rows = [("Bilder der Reihe", True, f"{len(FRAMES)} in {SESS.frames_dir}"),
        ("Referenz und WCS", RES.source == "ASTAP", f"{REF.name}: {RES.source}, {SUMM.get('scale_arcsec', '?')} ″/px"),
        ("WCS gegen Gaia", bool(QUAL.get("checked")) and QUAL.get("verdict") == "OK", QUAL.get("verdict") or QUAL.get("note", "")),
        ("Comps", FIELD["counts"]["comp"] >= 4, f"{FIELD['counts']['comp']} Comps, {FIELD['counts']['check']} Check, Sequenz-ID {FIELD['record']['vsp'].get('chartid') or '—'}"),
        ("Gespeichert", True, str(FIELD["path"]))]
html = "".join(f'<tr><td style="padding:3px 10px;color:{"#4caf50" if ok else "#ff9800"};font-weight:bold">{"✓" if ok else "⚠"}</td><td style="padding:3px 10px">{a}</td><td style="padding:3px 10px;color:#888">{b}</td></tr>' for a, ok, b in rows)
display(HTML(f'<table style="border-collapse:collapse;font-family:monospace;font-size:13px"><tr style="background:#1e1e2e;color:#cdd6f4"><th colspan="3" style="padding:6px 14px;text-align:left">Ergebnis Vorbereitung</th></tr>{html}</table>'))
print("→ weiter mit 02_messung.ipynb")